In [4]:
import os

os.getcwd()

'/Users/josephpatterson/Projects/chicago-permits-analysis'

In [5]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "docker-compose.yml").exists():
    root = root.parent

df = pd.read_csv(root / "data/raw/building_permits.csv", low_memory=False)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 852143 entries, 0 to 852142
Columns: 117 entries, ID to LOCATION
dtypes: float64(10), str(107)
memory usage: 760.7 MB


In [6]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "pct_filled": (df.notna().mean() * 100).round(1),
    "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
})
print(summary.to_string())

                           dtype  pct_filled                                                                                                                                                                                                              example
ID                           str       100.0                                                                                                                                                                                                              2521709
PERMIT#                      str       100.0                                                                                                                                                                                                            100519123
PERMIT_STATUS                str        64.4                                                                                                                                                                                      

In [7]:
# 1. Are IDs unique, and what do the non-numeric ones look like?
print("Duplicate IDs:", df["ID"].duplicated().sum())
bad_id = df.loc[~df["ID"].str.fullmatch(r"\d+", na=False), "ID"]
print("Non-numeric IDs:", len(bad_id), bad_id.head(10).tolist())

# 2. Processing times that aren't plain numbers
pt = pd.to_numeric(df["PROCESSING_TIME"].str.replace(",", ""), errors="coerce")
print("\nNon-numeric processing times:", pt.isna().sum())
print(df.loc[pt.isna(), "PROCESSING_TIME"].value_counts().head(10))
print("Negative processing times:", (pt < 0).sum())

# 3. Dates
app = pd.to_datetime(df["APPLICATION_START_DATE"], format="%m/%d/%Y", errors="coerce")
iss = pd.to_datetime(df["ISSUE_DATE"], format="%m/%d/%Y", errors="coerce")
print("\nUnparseable dates (application, issue):", app.isna().sum(), iss.isna().sum())
print("Date range:", app.min(), "to", iss.max())
print("Issued before application date:", (iss < app).sum())

# 4. Reported cost
cost = pd.to_numeric(df["REPORTED_COST"].str.replace(r"[$,]", "", regex=True), errors="coerce")
print("\nZero-cost permits:", (cost == 0).sum())
print(cost.describe())

# 5. Longest values vs. the column sizes in the schema
limits = {"PERMIT_TYPE": 100, "REVIEW_TYPE": 100, "WORK_TYPE": 100,
          "PERMIT_MILESTONE": 200, "STREET_NAME": 100, "PERMIT#": 20}
print()
for col, limit in limits.items():
    print(f"{col}: longest {df[col].str.len().max()} (limit {limit})")
name_cols = [f"CONTACT_{i}_NAME" for i in range(1, 16)]
print("Longest contact name:", df[name_cols].apply(lambda s: s.str.len().max()).max(), "(limit 255)")

Duplicate IDs: 0
Non-numeric IDs: 53153 ['N2886972', 'N2886515', 'N2884482', 'N2072729', 'N2112277', 'N2154968', 'N2160251', 'N2557751', 'N2633902', 'N2671387']

Non-numeric processing times: 349
Series([], Name: count, dtype: int64)
Negative processing times: 81

Unparseable dates (application, issue): 349 0
Date range: 2000-10-06 00:00:00 to 2026-11-03 00:00:00
Issued before application date: 81

Zero-cost permits: 21120
count    8.136240e+05
mean     2.349415e+05
std      1.762568e+07
min     -1.000000e+03
25%      9.080000e+02
50%      5.000000e+03
75%      2.500000e+04
max      1.000000e+10
Name: REPORTED_COST, dtype: float64

PERMIT_TYPE: longest 31 (limit 100)
REVIEW_TYPE: longest 30.0 (limit 100)
WORK_TYPE: longest 34.0 (limit 100)
PERMIT_MILESTONE: longest 74.0 (limit 200)
STREET_NAME: longest 45.0 (limit 100)
PERMIT#: longest 10 (limit 20)
Longest contact name: 80.0 (limit 255)
